# Stage 03 - Bronze Ingest

Preserve the simulated observed, operational, and predicted lanes with source fidelity. The PostgreSQL mirror is preferred for operational tables; checked-in CSV batches are the deterministic fallback.

> All identifiers, locations, measures, and labels are fictional demo values.

In [ ]:
from pyspark.sql import functions as F

source_files = {
    "observed": "Files/shared/integrated-test-data/projections/medallion/observed_test_events.jsonl",
    "operations": "Files/shared/integrated-test-data/projections/medallion/operations_snapshot.csv",
    "baseline": "Files/shared/integrated-test-data/projections/medallion/emulated_test_baseline.csv",
}

table_names = {
    "observed": "bronze_observed_test_events",
    "operations": "bronze_operations_snapshot",
    "baseline": "bronze_emulated_test_baseline",
    "metadata": "bronze_ingest_metadata",
}

for lane_name, source_file in source_files.items():
    print(f"Shared release input: {lane_name} -> {source_file}")

In [ ]:
def add_bronze_metadata(df, lane_name, source_file):
    payload_columns = [F.col(column_name) for column_name in df.columns]
    return (
        df.withColumn("bronze_lane", F.lit(lane_name))
        .withColumn("bronze_source_file", F.lit(source_file))
        .withColumn("bronze_loaded_at_utc", F.current_timestamp())
        .withColumn("bronze_record_sha256", F.sha2(F.to_json(F.struct(*payload_columns)), 256))
    )

observed_raw_df = spark.read.option("multiLine", "false").json(source_files["observed"])
operations_raw_df = spark.read.option("header", "true").csv(source_files["operations"])
baseline_raw_df = spark.read.option("header", "true").csv(source_files["baseline"])

bronze_observed_df = add_bronze_metadata(observed_raw_df, "observed_events", source_files["observed"])
bronze_operations_df = add_bronze_metadata(operations_raw_df, "operations_snapshot", source_files["operations"])
bronze_baseline_df = add_bronze_metadata(baseline_raw_df, "emulated_baseline", source_files["baseline"])

print(f"Observed rows landed: {bronze_observed_df.count()}")
print(f"Operations rows landed: {bronze_operations_df.count()}")
print(f"Baseline rows landed: {bronze_baseline_df.count()}")

In [ ]:
bronze_tables = [
    ("observed_events", source_files["observed"], table_names["observed"], bronze_observed_df, "Checked-in JSONL stands in for a Kafka or Eventstream lane."),
    ("operations_snapshot", source_files["operations"], table_names["operations"], bronze_operations_df, "Checked-in CSV stands in for a mirrored operations snapshot."),
    ("emulated_baseline", source_files["baseline"], table_names["baseline"], bronze_baseline_df, "Checked-in CSV stands in for a committed ADLS baseline extract."),
]

ingest_summary_rows = []
for lane_name, source_file, table_name, frame, stand_in_note in bronze_tables:
    row_count = frame.count()
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    ingest_summary_rows.append((lane_name, source_file, table_name, row_count, stand_in_note))

bronze_ingest_metadata_df = spark.createDataFrame(
    ingest_summary_rows,
    ["bronze_lane", "bronze_source_file", "target_table", "row_count", "stand_in_note"],
).withColumn("metadata_loaded_at_utc", F.current_timestamp())

(
    bronze_ingest_metadata_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_names["metadata"])
)

print("Bronze tables are ready for canonicalization.")

## Bronze review

Bronze is intentionally source preserving. The malformed event remains present here, and duplicate messages are still visible. Silver will decide what becomes canonical and what moves to quarantine.

In [ ]:
spark.table(table_names["metadata"]).orderBy("bronze_lane").show(truncate=False)

spark.table(table_names["observed"]).select(
    "event_id",
    "event_type",
    "site_id",
    "source_instance_id",
    "ingest_time_utc",
    "bronze_record_sha256",
).orderBy("event_type", "event_id", "ingest_time_utc").show(truncate=False)

spark.table(table_names["observed"]).filter(F.col("event_id") == "malformed-event-001").show(truncate=False)

spark.table(table_names["baseline"]).orderBy("planned_event_time_utc").show(truncate=False)
spark.table(table_names["operations"]).orderBy("reported_time_local").show(truncate=False)